In [1]:
import numpy as np
import pandas as pd
import h5py

with open(
    "../data/raw/sensor_graph/graph_sensor_ids.txt",
    "r"
) as f:
    graph_sensor_ids = [
        sensor_id.strip()
        for sensor_id in f.read().split(",")
        if sensor_id.strip()
    ]

print("Number of graph sensors:", len(graph_sensor_ids))
print("First 10 graph sensors:", graph_sensor_ids[:10])

Number of graph sensors: 207
First 10 graph sensors: ['773869', '767541', '767542', '717447', '717446', '717445', '773062', '767620', '737529', '717816']


In [2]:

with h5py.File("../data/raw/metr-la.h5", "r") as f:
    dataset_sensor_ids = [
        sensor_id.decode("utf-8")
        for sensor_id in f["df"]["axis0"][:]
    ]

print("Number of dataset sensors:", len(dataset_sensor_ids))
print("First 10 dataset sensors:", dataset_sensor_ids[:10])

Number of dataset sensors: 207
First 10 dataset sensors: ['773869', '767541', '767542', '717447', '717446', '717445', '773062', '767620', '737529', '717816']


In [3]:

same_order = graph_sensor_ids == dataset_sensor_ids
same_set = set(graph_sensor_ids) == set(dataset_sensor_ids)

print("Same number of sensors:", 
      len(graph_sensor_ids) == len(dataset_sensor_ids))

print("Same sensor set:", same_set)

print("Same sensor order:", same_order)

Same number of sensors: True
Same sensor set: True
Same sensor order: True


In [4]:

distance_df = pd.read_csv(
    "../data/raw/sensor_graph/distances_la_2012.csv"
)

print("Distance data shape:", distance_df.shape)
print("\nColumns:")
print(distance_df.columns.tolist())

print("\nFirst 10 rows:")
display(distance_df.head(10))

Distance data shape: (295374, 3)

Columns:
['from', 'to', 'cost']

First 10 rows:


,from,to,cost
0,1201054,1201054,0.0
1,1201054,1201066,2610.9
2,1201054,1201076,2822.7
3,1201054,1201087,2911.5
4,1201054,1201100,7160.1
5,1201054,1201112,2656.5
6,1201054,1201118,3417.2
7,1201054,1201125,3659.9
8,1201054,1201145,6168.4
9,1201054,1201159,2651.8


In [5]:
print("\nData types:")
print(distance_df.dtypes)

print("\nMissing values:")
print(distance_df.isna().sum())

print("\nDistance statistics:")
print(distance_df.describe())


Data types:
from      int64
to        int64
cost    float64
dtype: object

Missing values:
from    0
to      0
cost    0
dtype: int64

Distance statistics:
               from            to           cost
count  2.953740e+05  2.953740e+05  295374.000000
mean   8.899667e+05  8.898141e+05    7201.727881
std    2.060738e+05  2.059274e+05    2659.886587
min    7.158980e+05  7.158980e+05       0.000000
25%    7.182850e+05  7.182870e+05    5539.125000
50%    7.737840e+05  7.738070e+05    7730.300000
75%    1.202785e+06  1.202724e+06    9316.200000
max    1.219710e+06  1.219710e+06   21254.800000


In [6]:

graph_sensor_set = set(graph_sensor_ids)

distance_sensor_set = set(
    distance_df["from"].astype(str)
).union(
    set(distance_df["to"].astype(str))
)

matched_distance_ids = graph_sensor_set.intersection(
    distance_sensor_set
)

unmatched_graph_ids = graph_sensor_set - distance_sensor_set
unmatched_distance_ids = distance_sensor_set - graph_sensor_set

print("Graph sensor IDs:", len(graph_sensor_set))
print("Unique distance-file IDs:", len(distance_sensor_set))
print("Matched IDs:", len(matched_distance_ids))
print("Graph IDs without distance entries:", len(unmatched_graph_ids))
print("Distance IDs not in graph sensor list:", len(unmatched_distance_ids))

print("\nFirst 10 unmatched graph IDs:")
print(sorted(unmatched_graph_ids)[:10])

print("\nFirst 10 distance IDs:")
print(sorted(distance_sensor_set)[:10])

Graph sensor IDs: 207
Unique distance-file IDs: 4106
Matched IDs: 207
Graph IDs without distance entries: 0
Distance IDs not in graph sensor list: 3899

First 10 unmatched graph IDs:
[]

First 10 distance IDs:
['1201054', '1201066', '1201076', '1201087', '1201100', '1201112', '1201118', '1201125', '1201145', '1201159']


In [7]:

graph_distance_df = distance_df[
    distance_df["from"].astype(str).isin(graph_sensor_set)
    & distance_df["to"].astype(str).isin(graph_sensor_set)
].copy()

print("Original distance rows:", len(distance_df))
print("Distance rows involving only our 207 sensors:", len(graph_distance_df))

print("\nFirst 10 relevant distance records:")
display(graph_distance_df.head(10))

Original distance rows: 295374
Distance rows involving only our 207 sensors: 11753

First 10 relevant distance records:


,from,to,cost
92905,716328,716328,0.0
92906,716328,716331,4123.8
92907,716328,716337,5179.6
92908,716328,716339,7245.5
92913,716328,716939,4785.1
92915,716328,716941,5662.4
92916,716328,716942,5398.7
92917,716328,716943,5413.2
92920,716328,716949,7767.2
92921,716328,716951,5852.6


In [9]:

num_sensors = len(graph_sensor_ids)

sensor_id_to_index = {
    sensor_id: index
    for index, sensor_id in enumerate(graph_sensor_ids)
}

distance_matrix = np.full(
    (num_sensors, num_sensors),
    np.inf,
    dtype=np.float32
)

for _, row in graph_distance_df.iterrows():
    from_id = str(row["from"])
    to_id = str(row["to"])
    distance = float(row["cost"])

    if from_id in sensor_id_to_index and to_id in sensor_id_to_index:
        i = sensor_id_to_index[from_id]
        j = sensor_id_to_index[to_id]
        distance_matrix[i, j] = distance

print("Distance matrix shape:", distance_matrix.shape)

finite_distances = distance_matrix[np.isfinite(distance_matrix)]

print("Finite distance entries:", len(finite_distances))
print("Minimum distance:", finite_distances.min())
print("Maximum distance:", finite_distances.max())
print("Mean distance:", finite_distances.mean())

print("\nMissing distance entries:", np.isinf(distance_matrix).sum())

Distance matrix shape: (207, 207)
Finite distance entries: 0


ValueError: zero-size array to reduction operation minimum which has no identity

In [10]:

print("graph_sensor_ids example:")
print(graph_sensor_ids[:5])

print("\nType of graph_sensor_ids[0]:")
print(type(graph_sensor_ids[0]))

print("\ndistance_df dtypes:")
print(distance_df[["from", "to", "cost"]].dtypes)

print("\nGraph distance dataframe dtypes:")
print(graph_distance_df[["from", "to", "cost"]].dtypes)

print("\nFirst distance row:")
print(graph_distance_df.iloc[0])

print("\nTypes in first distance row:")
print(type(graph_distance_df.iloc[0]["from"]))
print(type(graph_distance_df.iloc[0]["to"]))

print("\nFirst distance ID as string:")
print(str(graph_distance_df.iloc[0]["from"]))

print("\nDoes first distance ID exist in our sensor dictionary?")
print(str(graph_distance_df.iloc[0]["from"]) in sensor_id_to_index)

print("\nDirect integer lookup if applicable:")
try:
    print(graph_distance_df.iloc[0]["from"] in sensor_id_to_index)
except Exception as e:
    print("Error:", e)

graph_sensor_ids example:
['773869', '767541', '767542', '717447', '717446']

Type of graph_sensor_ids[0]:
<class 'str'>

distance_df dtypes:
from      int64
to        int64
cost    float64
dtype: object

Graph distance dataframe dtypes:
from      int64
to        int64
cost    float64
dtype: object

First distance row:
from    716328.0
to      716328.0
cost         0.0
Name: 92905, dtype: float64

Types in first distance row:
<class 'numpy.float64'>
<class 'numpy.float64'>

First distance ID as string:
716328.0

Does first distance ID exist in our sensor dictionary?
False

Direct integer lookup if applicable:
False


In [11]:

num_sensors = len(graph_sensor_ids)

sensor_id_to_index = {
    sensor_id: index
    for index, sensor_id in enumerate(graph_sensor_ids)
}

distance_matrix = np.full(
    (num_sensors, num_sensors),
    np.inf,
    dtype=np.float32
)

for row in graph_distance_df.itertuples(index=False):
    from_id = str(int(row[0]))
    to_id = str(int(row[1]))
    distance = float(row[2])

    if from_id in sensor_id_to_index and to_id in sensor_id_to_index:
        i = sensor_id_to_index[from_id]
        j = sensor_id_to_index[to_id]
        distance_matrix[i, j] = distance

finite_distances = distance_matrix[np.isfinite(distance_matrix)]

print("Distance matrix shape:", distance_matrix.shape)
print("Finite distance entries:", len(finite_distances))
print("Minimum distance:", finite_distances.min())
print("Maximum distance:", finite_distances.max())
print("Mean distance:", finite_distances.mean())
print("Missing distance entries:", np.isinf(distance_matrix).sum())

Distance matrix shape: (207, 207)
Finite distance entries: 11753
Minimum distance: 0.0
Maximum distance: 11895.3
Mean distance: 6899.1567
Missing distance entries: 31096


In [12]:

finite_mask = np.isfinite(distance_matrix)

connections_per_sensor = finite_mask.sum(axis=1)

print("Sensors:", num_sensors)

print("\nSensors with zero connections:",
      np.sum(connections_per_sensor == 0))

print("Minimum connections for a sensor:",
      connections_per_sensor.min())

print("Maximum connections for a sensor:",
      connections_per_sensor.max())

print("Average connections per sensor:",
      connections_per_sensor.mean())

diagonal = np.diag(distance_matrix)

print("\nDiagonal finite entries:",
      np.isfinite(diagonal).sum())

print("Diagonal zero entries:",
      np.sum(diagonal == 0))

finite_both = np.isfinite(distance_matrix) & np.isfinite(distance_matrix.T)

symmetric_pairs = np.sum(
    np.isclose(
        distance_matrix[finite_both],
        distance_matrix.T[finite_both]
    )
)

total_comparable_pairs = np.sum(finite_both)

print("\nComparable distance pairs:", total_comparable_pairs)
print("Symmetric distance pairs:", symmetric_pairs)

Sensors: 207

Sensors with zero connections: 0
Minimum connections for a sensor: 8
Maximum connections for a sensor: 112
Average connections per sensor: 56.77777777777778

Diagonal finite entries: 207
Diagonal zero entries: 207

Comparable distance pairs: 8563
Symmetric distance pairs: 227


In [13]:

finite_distances = distance_matrix[np.isfinite(distance_matrix)]

distance_std = finite_distances.std()

print("Distance standard deviation:", distance_std)

adjacency_matrix = np.exp(
    -np.square(distance_matrix / distance_std)
)

NORMALIZED_K = 0.1

adjacency_matrix[adjacency_matrix < NORMALIZED_K] = 0

adjacency_matrix[~np.isfinite(distance_matrix)] = 0

print("\nAdjacency matrix shape:", adjacency_matrix.shape)

print("Minimum weight:", adjacency_matrix.min())
print("Maximum weight:", adjacency_matrix.max())

print("Non-zero connections:",
      np.count_nonzero(adjacency_matrix))

print("Graph density:",
      np.count_nonzero(adjacency_matrix) / adjacency_matrix.size)

print("Zero connections:",
      np.sum(adjacency_matrix == 0))

Distance standard deviation: 2584.4502

Adjacency matrix shape: (207, 207)
Minimum weight: 0.0
Maximum weight: 1.0
Non-zero connections: 1722
Graph density: 0.04018763565077365
Zero connections: 41127


In [14]:

print("Adjacency matrix shape:", adjacency_matrix.shape)

print("Contains NaN:", np.isnan(adjacency_matrix).any())
print("Contains Inf:", np.isinf(adjacency_matrix).any())

print("Minimum weight:", adjacency_matrix.min())
print("Maximum weight:", adjacency_matrix.max())

self_connections = np.diag(adjacency_matrix)

print("\nSelf-connections:", np.sum(self_connections > 0))

non_self_connections = (
    np.count_nonzero(adjacency_matrix)
    - np.count_nonzero(np.diag(adjacency_matrix))
)

print("Non-self connections:", non_self_connections)

neighbours_per_sensor = (
    np.count_nonzero(adjacency_matrix, axis=1) - 1
)

print("\nMinimum non-self neighbours:",
      neighbours_per_sensor.min())

print("Maximum non-self neighbours:",
      neighbours_per_sensor.max())

print("Average non-self neighbours:",
      neighbours_per_sensor.mean())

Adjacency matrix shape: (207, 207)
Contains NaN: False
Contains Inf: False
Minimum weight: 0.0
Maximum weight: 1.0

Self-connections: 207
Non-self connections: 1515

Minimum non-self neighbours: 0
Maximum non-self neighbours: 18
Average non-self neighbours: 7.318840579710145


In [15]:
import os
import pickle

graph_output_dir = "../data/processed"

os.makedirs(graph_output_dir, exist_ok=True)

graph_path = os.path.join(
    graph_output_dir,
    "adjacency_matrix.npy"
)

np.save(
    graph_path,
    adjacency_matrix.astype(np.float32)
)

sensor_order_path = os.path.join(
    graph_output_dir,
    "graph_sensor_ids.txt"
)

with open(sensor_order_path, "w") as f:
    f.write(",".join(graph_sensor_ids))

print("Saved adjacency matrix:")
print(graph_path)

print("\nSaved sensor ordering:")
print(sensor_order_path)

Saved adjacency matrix:
../data/processed/adjacency_matrix.npy

Saved sensor ordering:
../data/processed/graph_sensor_ids.txt


In [16]:

import numpy as np

loaded_adjacency = np.load(
    "../data/processed/adjacency_matrix.npy"
)

with open("../data/processed/graph_sensor_ids.txt", "r") as f:
    loaded_sensor_ids = [
        sensor_id.strip()
        for sensor_id in f.read().split(",")
        if sensor_id.strip()
    ]

print("Loaded adjacency shape:", loaded_adjacency.shape)
print("Loaded sensor count:", len(loaded_sensor_ids))

print("Contains NaN:", np.isnan(loaded_adjacency).any())
print("Contains Inf:", np.isinf(loaded_adjacency).any())

print(
    "Matrix identical to original:",
    np.array_equal(loaded_adjacency, adjacency_matrix)
)

print(
    "Sensor ordering identical:",
    loaded_sensor_ids == graph_sensor_ids
)

Loaded adjacency shape: (207, 207)
Loaded sensor count: 207
Contains NaN: False
Contains Inf: False
Matrix identical to original: True
Sensor ordering identical: True
